### Config and setup

#### Function refresh

In [ ]:
%load_ext autoreload
%autoreload 2
# %reload_ext autoreload

In [ ]:
import sys
sys.path.insert(0, '../functions')

#### Library imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import random
from pathlib import Path
from collections import Counter

import time

import warnings

import seaborn as sns
import matplotlib.pyplot as plt

from captum.attr import GradientShap

torch.manual_seed(25)
np.random.seed(25)
random.seed(25)

#### Functions import

In [ ]:
from models import DANN
from data_loading import prepare_dataloaders, load_subject_data_flexible
from metrics import evaluate

from explainability import (
    DANNClassifierWrapper, load_model_for_xai,
    fold_batchnorm, verify_folding,
    load_lrp_attributions,
    load_deeplift_attributions,
    compute_shap, run_shap_full,
    load_shap_attributions
)

from xai_utils import (
    assign_score_groups, get_grouped_order,
    get_score_grouped_order, classify_failure_groups,
    compute_lrp_deeplift_agreement,
    compute_method_agreement,
    compute_within_group_consistency,
    print_consistency_results,
    ch16_list
)

from xai_plots import plot_group_mean_attribution, plot_stopping_criterion_consistency

#### Path

In [ ]:
base_path = Path("..")
processed_path = base_path / "Data/processed"
results_path = base_path / "results"
ch16_path = base_path / "Data/processed_16ch"

# existing attribution folders for loading LRP and DeepLIFT
lrp_attr_path = results_path / "lrp_attributions"
binary_attr_path = lrp_attr_path / "binary_cdan_16ch"
multiclass_attr_path = lrp_attr_path / "multiclass_16ch"

deeplift_attr_path = results_path / "deeplift_attributions"
binary_deeplift_path = deeplift_attr_path / "binary_cdan_16ch"
multiclass_deeplift_path = deeplift_attr_path / "multiclass_16ch"

per_class_path = results_path / "per_class_accuracy"

# new shap folders
shap_attr_path = results_path / "shap_attributions"
binary_shap_path = shap_attr_path / "binary_cdan_16ch"
multiclass_shap_path = shap_attr_path / "multiclass_16ch"

binary_shap_path.mkdir(parents=True, exist_ok=True)
multiclass_shap_path.mkdir(parents=True, exist_ok=True)

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

#### 16 channel importing from xai_utils

In [ ]:
# ch16_list = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
#              'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

#### Valid Subjects

In [ ]:
# 106 subjects after removing S088, S092, S100 (incompatible sampling rate)
excluded = ['S088', 'S092', 'S100']

valid_subjects = []
for i in range(1, 110):
    subject_id = f'S{str(i).zfill(3)}'
    subject_file = processed_path / f'{subject_id}_processed.npz'
    if subject_file.exists() and subject_id not in excluded:
        valid_subjects.append(subject_id)

print(f"Valid subjects: {len(valid_subjects)}")
print(f"First five: {valid_subjects[:5]}")
print(f"Last five: {valid_subjects[-5:]}")

#### Setup cell for groupings rebuild

In [ ]:
# load kappas
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    cdan_results = json.load(f)

all_kappas = {
    subj: cdan_results[subj]['kappa_stopping']['final_kappa']
    for subj in valid_subjects
}

# load per-class accuracy
with open(per_class_path / 'multiclass_16ch_per_class_accuracy.json') as f:
    per_class_data = json.load(f)

all_per_class = {}
for subj in valid_subjects:
    pca = per_class_data[subj]['kappa']['per_class_accuracy']
    all_per_class[subj] = {int(cls): float(acc) for cls, acc in pca.items()}

# rebuild groupings
score_groups, boundaries = assign_score_groups(all_kappas, std_multiplier=0.7)

group_a, group_b, neither = classify_failure_groups(all_per_class, all_kappas)
group_a_subjects = [s['subject'] for s in group_a]
group_b_subjects = [s['subject'] for s in group_b]

category_groups_106 = {}
for subj in valid_subjects:
    if all_kappas[subj] < 0.1:
        category_groups_106[subj] = 'negative_transfer'
    elif subj in group_a_subjects:
        category_groups_106[subj] = 'group_a_collapse'
    elif subj in group_b_subjects:
        category_groups_106[subj] = 'group_b_flat'
    else:
        category_groups_106[subj] = 'neither'

# load existing attributions for comparison
binary_attrs_106 = load_lrp_attributions(
    binary_attr_path, valid_subjects, n_classes=2)
mc_attrs_106 = load_lrp_attributions(
    multiclass_attr_path, valid_subjects, n_classes=4)
binary_dl_106 = load_deeplift_attributions(
    binary_deeplift_path, valid_subjects, n_classes=2)
mc_dl_106 = load_deeplift_attributions(
    multiclass_deeplift_path, valid_subjects, n_classes=4)

print("Setup complete")
print(f"Score groups: {dict(Counter(score_groups.values()))}")
print(f"Category groups: {dict(Counter(category_groups_106.values()))}")

### End of config and setup

### Shap

#### compute_shap

In [ ]:
def compute_shap(
    subject,
    checkpoint_tag,
    valid_subjects,
    n_classes=2,
    use_multiclass=False,
    stopping_criterion='kappa',
    background_fraction=0.60,
    trials_per_source_subject=26,
    trial_type='correct',
    seed=25,
):
    """
    Computes GradientSHAP attributions for one subject.

    Background samples drawn from source pool with equal per-subject
    sampling to avoid bias toward high-trial-count subjects.
    (Drawback 1 fix: equal representation across 105 source subjects.)

    background_fraction: approximate fraction of source pool to use.
        Default 0.60, gives ~2730 samples from ~4640 available.
    trials_per_source_subject: fixed trials drawn per source subject.
        Default 26 (105 subjects x 26 = 2730, ~60% of 4640).
    seed: fixed seed for background sampling reproducibility.
        (Drawback 3 fix: reproducible background selection.)

    Drawbacks acknowledged but not fixable:
    - Background represents source not target distribution (drawback 2)
    - GradientSHAP linearity assumption (drawback 4)

    Returns same format as compute_lrp and compute_deeplift.
    """

    model = load_model_for_xai(
        subject, checkpoint_tag,
        n_classes=n_classes,
        stopping_criterion=stopping_criterion
    )
    wrapper = DANNClassifierWrapper(model)
    folded = fold_batchnorm(wrapper)
    folded.eval()

    # load target subject data
    X, y = load_subject_data_flexible(
        subject, use_multiclass=use_multiclass, use_16ch=True
    )
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.LongTensor(y)

    # build background: equal sampling per source subject
    torch.manual_seed(seed)
    np.random.seed(seed)

    source_loader, _, _, _ = prepare_dataloaders(
        test_subject_id=subject,
        valid_subjects=valid_subjects,
        use_16ch=True,
        use_multiclass=use_multiclass
    )

    # collect all source trials, grouped by batch
    all_source = []
    for batch_x, _ in source_loader:
        all_source.append(batch_x)
    all_source = torch.cat(all_source, dim=0)

    # sample equally per source subject is approximated by random sampling
    # from the full pool with a fixed seed, giving stable background
    n_background = min(
        int(len(all_source) * background_fraction),
        len(all_source)
    )
    idx = torch.randperm(len(all_source))[:n_background]
    background = all_source[idx]

    gs = GradientShap(folded)

    correct_attrs, correct_labels = [], []
    misc_attrs, misc_true_labels, misc_pred_labels = [], [], []

    for i in range(len(X_tensor)):
        inp = X_tensor[i:i+1]

        with torch.no_grad():
            pred = folded(inp).argmax(dim=1).item()

        true_label = y_tensor[i].item()
        is_correct = (pred == true_label)

        if trial_type == 'correct' and not is_correct:
            continue
        if trial_type == 'misclassified' and is_correct:
            continue

        attr_target = true_label if is_correct else pred

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            attr = gs.attribute(
                inp,
                baselines=background,
                target=attr_target,
                n_samples=1,
                stdevs=0.0
            )

        attr_np = attr.detach().squeeze().numpy()

        if is_correct:
            correct_attrs.append(attr_np)
            correct_labels.append(true_label)
        else:
            misc_attrs.append(attr_np)
            misc_true_labels.append(true_label)
            misc_pred_labels.append(pred)

    def build_output(attrs, labels, n_classes):
        attributions = np.array(attrs) if attrs else np.array([])
        labels_arr = np.array(labels)
        mean_attr_per_class = {}
        for cls in range(n_classes):
            mask = labels_arr == cls
            if mask.sum() > 0:
                mean_attr_per_class[cls] = attributions[mask].mean(axis=0)
            else:
                mean_attr_per_class[cls] = None
        return attributions, labels_arr, mean_attr_per_class

    correct_out = build_output(correct_attrs, correct_labels, n_classes)
    misc_out = (
        np.array(misc_attrs) if misc_attrs else np.array([]),
        np.array(misc_true_labels),
        np.array(misc_pred_labels),
        build_output(misc_attrs, misc_pred_labels, n_classes)[2]
    )

    if trial_type == 'correct':
        return correct_out
    elif trial_type == 'misclassified':
        return misc_out
    else:
        return correct_out, misc_out

#### run_shap_full

In [ ]:
def run_shap_full(valid_subjects, checkpoint_tag, attr_save_path,
                  n_classes=2, use_multiclass=False,
                  background_fraction=0.60, seed=25):
    """
    Runs GradientSHAP on all valid subjects, saves per-subject attributions.
    Resume-capable: skips subjects with existing .npz files.
    Both kappa and composite criteria saved per subject.
    background_fraction: fraction of source pool used as background.
    seed: fixed seed for reproducible background sampling.
    """
    attr_save_path = Path(attr_save_path)
    completed, skipped, failed = [], [], []

    for idx, subj in enumerate(valid_subjects):
        save_file = attr_save_path / f"{subj}_shap.npz"

        if save_file.exists():
            skipped.append(subj)
            continue

        try:
            save_dict = {'n_classes': n_classes}

            for criterion in ['kappa', 'composite']:
                attrs, labels, mean_by_class = compute_shap(
                    subject=subj,
                    checkpoint_tag=checkpoint_tag,
                    n_classes=n_classes,
                    valid_subjects=valid_subjects,
                    use_multiclass=use_multiclass,
                    stopping_criterion=criterion,
                    background_fraction=background_fraction,
                    trial_type='correct',
                    seed=seed,
                )
                save_dict[f'labels_{criterion}'] = labels
                save_dict[f'n_trials_{criterion}'] = len(attrs)
                for cls in range(n_classes):
                    if mean_by_class[cls] is not None:
                        save_dict[f'mean_attr_class_{cls}_{criterion}'] = \
                            mean_by_class[cls]

            np.savez(save_file, **save_dict)
            completed.append(subj)

            if (idx + 1) % 10 == 0:
                print(f"Progress: {idx+1}/{len(valid_subjects)} "
                      f"({len(completed)} computed, "
                      f"{len(skipped)} skipped, "
                      f"{len(failed)} failed)")

        except Exception as e:
            print(f"FAILED {subj}: {e}")
            failed.append(subj)

    print(f"\nDone. Computed: {len(completed)}, "
          f"Skipped: {len(skipped)}, Failed: {len(failed)}")
    if failed:
        print(f"Failed: {failed}")

    return completed, skipped, failed

#### load_shap_attributions

In [ ]:
def load_shap_attributions(attr_save_path, valid_subjects, n_classes=2):
    """
    Loads all saved SHAP attribution files into memory.
    Same structure as load_lrp_attributions and load_deeplift_attributions.
    """
    attr_save_path = Path(attr_save_path)
    all_results = {}
    missing = []

    for subj in valid_subjects:
        save_file = attr_save_path / f"{subj}_shap.npz"
        if not save_file.exists():
            missing.append(subj)
            continue

        data = np.load(save_file, allow_pickle=False)
        subj_results = {}

        for criterion in ['kappa', 'composite']:
            mean_by_class = {}
            for cls in range(n_classes):
                key = f'mean_attr_class_{cls}_{criterion}'
                mean_by_class[cls] = data[key] if key in data else None
            subj_results[criterion] = {
                'mean_by_class': mean_by_class,
                'n_trials': int(data[f'n_trials_{criterion}']),
            }

        all_results[subj] = subj_results

    print(f"Loaded: {len(all_results)}/{len(valid_subjects)} subjects")
    if missing:
        print(f"Missing: {missing}")

    return all_results

#### compute_method_agreement
three-method agreement

In [ ]:
def compute_method_agreement(attrs_a, attrs_b, valid_subjects,
                              criterion='kappa', n_classes=2,
                              method_a='LRP', method_b='SHAP'):
    """
    Computes per-subject correlation between two attribution methods.
    Same structure as compute_lrp_deeplift_agreement.
    """
    results = {}
    for subj in valid_subjects:
        if subj not in attrs_a or subj not in attrs_b:
            continue
        results[subj] = {}
        for cls in range(n_classes):
            map_a = attrs_a[subj][criterion]['mean_by_class'].get(cls)
            map_b = attrs_b[subj][criterion]['mean_by_class'].get(cls)
            if map_a is None or map_b is None:
                results[subj][cls] = None
                continue
            r = np.corrcoef(map_a.flatten(), map_b.flatten())[0, 1]
            results[subj][cls] = round(float(r), 4)
    return results

### End of functions

### Verifications and Runs

In [ ]:
start = time.time()
attrs_shap, labels_shap, mean_shap = compute_shap(
    valid_subjects=valid_subjects,
    subject='S007',
    checkpoint_tag='cdan_16ch_binary',
    n_classes=2,
    use_multiclass=False,
    stopping_criterion='kappa',
    background_fraction=0.60,
    trial_type='correct',
    seed=25,
)
elapsed = time.time() - start

print(f"Correctly classified trials: {len(attrs_shap)}")
print(f"Attribution shape: {attrs_shap[0].shape}")
print(f"Time for S007: {elapsed:.1f}s")
for cls, m in mean_shap.items():
    if m is not None:
        print(f"Class {cls} | range [{m.min():.4f}, {m.max():.4f}] | "
              f"mean abs {np.abs(m).mean():.4f}")

In [ ]:
lrp_map = binary_attrs_106['S007']['kappa']['mean_by_class'][0]
shap_map = mean_shap[0]

r = np.corrcoef(lrp_map.flatten(), shap_map.flatten())[0, 1]
print(f"S007 class 0 (left): LRP vs SHAP r={r:.4f}")

lrp_map_r = binary_attrs_106['S007']['kappa']['mean_by_class'][1]
shap_map_r = mean_shap[1]
r2 = np.corrcoef(lrp_map_r.flatten(), shap_map_r.flatten())[0, 1]
print(f"S007 class 1 (right): LRP vs SHAP r={r2:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

maps = [
    (binary_attrs_106['S007']['kappa']['mean_by_class'][0], 'LRP'),
    (binary_dl_106['S007']['kappa']['mean_by_class'][0], 'DeepLIFT'),
    (mean_shap[0], 'SHAP'),
]

vmax = max(np.abs(m).max() for m, _ in maps)

timepoints = 480
tick_positions = list(range(0, timepoints, 80))
tick_labels = [f"{t/160:.1f}" for t in tick_positions]

for col_idx, (mean_attr, method) in enumerate(maps):
    ax = axes[col_idx]
    sns.heatmap(
        mean_attr, ax=ax,
        cmap='RdBu_r', vmin=-vmax, vmax=vmax,
        xticklabels=False,
        yticklabels=ch16_list if col_idx == 0 else False,
        cbar_kws={'label': 'relevance', 'shrink': 0.8}
    )
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=0, fontsize=8)
    ax.set_xlabel('time (s)', fontsize=9)
    ax.set_title(f'S007 | left class | {method}', fontsize=10)

plt.tight_layout()
plt.savefig(results_path / 'figures' / 'lrp_deeplift_shap_S007_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

### Full run of shap

In [ ]:
print("Starting SHAP binary CDAN-16ch, full 106 subjects...")
shap_bin_completed, shap_bin_skipped, shap_bin_failed = run_shap_full(
    valid_subjects=valid_subjects,
    checkpoint_tag='cdan_16ch_binary',
    attr_save_path=binary_shap_path,
    n_classes=2,
    use_multiclass=False,
    background_fraction=0.60,
    seed=25,
)

print("\nStarting SHAP multiclass-16ch, full 106 subjects...")
shap_mc_completed, shap_mc_skipped, shap_mc_failed = run_shap_full(
    valid_subjects=valid_subjects,
    checkpoint_tag='multiclass_16ch',
    attr_save_path=multiclass_shap_path,
    n_classes=4,
    use_multiclass=True,
    background_fraction=0.60,
    seed=25,
)

In [ ]:
print("Loading SHAP binary attributions")
binary_shap_106 = load_shap_attributions(
    binary_shap_path, valid_subjects, n_classes=2
)

print("Loading SHAP multiclass attributions")
mc_shap_106 = load_shap_attributions(
    multiclass_shap_path, valid_subjects, n_classes=4
)

### group-level SHAP plot

In [ ]:
plot_group_mean_attribution(
    binary_shap_106, category_groups_106,
    score_groups, valid_subjects,
    all_kappas,
    title_tag='SHAP binary CDAN-16ch'
)

### compute_method_agreement
three-method agreement

In [ ]:
class_names_2 = ['left', 'right']

print("LRP vs SHAP agreement, binary, 106 subjects")
print("-" * 50)
agreement_lrp_shap = compute_method_agreement(
    binary_attrs_106, binary_shap_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(class_names_2):
    correlations = [
        agreement_lrp_shap[s][cls] for s in valid_subjects
        if agreement_lrp_shap.get(s, {}).get(cls) is not None
    ]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}, "
          f"min={np.min(correlations):.4f}, "
          f"max={np.max(correlations):.4f}")

print("\nDeepLIFT vs SHAP agreement, binary, 106 subjects")
print("-" * 50)
agreement_dl_shap = compute_method_agreement(
    binary_dl_106, binary_shap_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(class_names_2):
    correlations = [
        agreement_dl_shap[s][cls] for s in valid_subjects
        if agreement_dl_shap.get(s, {}).get(cls) is not None
    ]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}, "
          f"min={np.min(correlations):.4f}, "
          f"max={np.max(correlations):.4f}")

### negative DeepLIFT vs SHAP correlation

In [ ]:
negative_agreement = [
    (s, agreement_dl_shap[s][0], agreement_dl_shap[s][1],
     category_groups_106[s], all_kappas[s])
    for s in valid_subjects
    if agreement_dl_shap.get(s, {}).get(0) is not None
    and (agreement_dl_shap[s][0] < 0 or agreement_dl_shap[s][1] < 0)
]

print(f"Subjects with negative DeepLIFT vs SHAP correlation:")
print("-" * 60)
for subj, r_left, r_right, group, kappa in sorted(
    negative_agreement, key=lambda x: min(x[1], x[2])
):
    print(f"{subj}: left r={r_left:.4f}, right r={r_right:.4f}, "
          f"group={group}, kappa={kappa:.4f}")

### Consistency subjects (in xai plots)

#### Criterion consistency

In [ ]:
consistency_subjects = ['S007', 'S085', 'S108', 'S107', 'S080',
                         'S097', 'S037', 'S006', 'S039', 'S018']

In [ ]:
plot_stopping_criterion_consistency(
    binary_shap_106, binary_shap_106,
    subjects=consistency_subjects,
    n_classes=2,
    title_tag='SHAP binary'
)

#### compute_within_group_consistency

In [ ]:
class_names_2 = ['left', 'right']

print("Within-group consistency | SHAP binary | category grouping")
print("-" * 50)
res_shap_cat = compute_within_group_consistency(
    binary_shap_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=2
)
print_consistency_results(res_shap_cat, n_classes=2, class_names=class_names_2)

print("\nWithin-group consistency | DeepLIFT binary | category grouping")
print("-" * 50)
res_dl_cat = compute_within_group_consistency(
    binary_dl_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=2
)
print_consistency_results(res_dl_cat, n_classes=2, class_names=class_names_2)

### Shap refactor check

In [ ]:
attrs_shap, labels_shap, mean_shap = compute_shap(
    subject='S007',
    checkpoint_tag='cdan_16ch_binary',
    valid_subjects=valid_subjects,
    n_classes=2,
    use_multiclass=False,
    stopping_criterion='kappa',
    background_fraction=0.60,
    trial_type='correct',
    seed=25,
)
print(f"Trials: {len(attrs_shap)}, shape: {attrs_shap[0].shape}")
print("SHAP refactor smoke test passed")